# PupilOnPole experiments

Each stage runs SuperTuxKart in worker processes, so this kernel never holds a race. Run the cells in order. Set `PYSTK_PROJECT_SRC` before starting Jupyter.

In [ ]:
from pupil_on_pole.config import DaggerConfig, DatasetConfig, EnvConfig, EvalConfig, RLPDConfig
from pupil_on_pole import experiments as ex

## Smoke test

One seed per method on a tiny budget, to check the whole chain in a few minutes.

In [ ]:
smoke = ex.create_experiment(
    "smoke",
    seeds=(1,),
    dagger_cfg=DaggerConfig(online_steps=10_000, checkpoint_every=2_000),
    rlpd_cfg=RLPDConfig(online_steps=10_000, checkpoint_every=2_000, learning_starts=1_000),
    workers=3,
)
ex.generate_dataset(smoke)
ex.train_experiment(smoke)
ex.evaluate_experiment(smoke)
smoke_results = ex.load_experiment(smoke)
ex.plot_experiment(smoke_results, smoke)
smoke_results["summary"]["main"]

## Pilot

Seed 1, both methods, 100k online steps. The budget B is the step where the curves flatten.

In [ ]:
pilot = ex.create_experiment("pilot", seeds=(1,), workers=3)
ex.generate_dataset(pilot)
ex.train_experiment(pilot)
ex.evaluate_experiment(pilot, held_out=False)
pilot_results = ex.load_experiment(pilot)
ex.plot_experiment(pilot_results, pilot)

## Main study

Seeds 1, 2 and 3 at budget B, then held-out evaluation and the tables and figures.

In [ ]:
BUDGET = 100_000  
main = ex.create_experiment(
    "main",
    seeds=(1, 2, 3),
    dagger_cfg=DaggerConfig(online_steps=BUDGET),
    rlpd_cfg=RLPDConfig(online_steps=BUDGET),
    workers=4,
)
ex.generate_dataset(main)
ex.train_experiment(main)
ex.evaluate_experiment(main)

In [ ]:
results = ex.load_experiment(main, budget=BUDGET)
ex.plot_experiment(results, main)
display(results["summary"]["main"], results["summary"]["welch"], results["summary"]["held_out"])